# Run FLFrameWork on Colab

Select **Runtime → Change runtime type → GPU**, then run the cells in order. Use a short run first to measure the assigned GPU's speed. This notebook runs one method at a time.

Source code can come from GitHub or the prepared project ZIP. Datasets and active CSV logs stay on the Colab disk; completed CSVs and figures are copied to Google Drive. If Python raises an error or you interrupt execution, the training cell also tries to copy its partial CSV. A disconnected or forcibly terminated runtime may lose work since the last copy. CSVs are not training checkpoints, so they cannot resume an interrupted run.


Check that Colab assigned an NVIDIA GPU and show its name before preparing the project.


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU under Runtime > Change runtime type, then reconnect.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

Clone your GitHub repository, or leave its URL empty and upload the prepared FLFrameWork-Colab.zip file.


In [ ]:
from pathlib import Path
import io
import os
import subprocess
import sys
import zipfile

REPOSITORY_URL = "https://github.com/s4meer-git/BPDC_FL_Framework.git"
PROJECT_ROOT = Path("/content/FLFrameWork")

if REPOSITORY_URL:
    if (PROJECT_ROOT / ".git").is_dir():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
else:
    from google.colab import files

    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith(".zip")]
    if len(archives) != 1:
        raise ValueError("Upload one FLFrameWork-Colab.zip file.")

    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        archive.extractall("/content")
    del uploaded

if not (PROJECT_ROOT / "main.py").is_file():
    raise FileNotFoundError("The project must contain /content/FLFrameWork/main.py.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project:", PROJECT_ROOT)

Install the small notebook dependencies and check torchvision while keeping Colab’s existing CUDA-enabled PyTorch installation.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "JPNotebooks/requirements.txt"],
    check=True,
)

import torchvision
import pandas as pd

print("torchvision:", torchvision.__version__)

Connect Google Drive so the method CSVs and analysis PDFs have a place to remain after this runtime ends.


In [ ]:
from google.colab import drive
import shutil

_drive_mount = "/content/drive"
drive.mount(_drive_mount)
DRIVE_ROOT = Path(_drive_mount) / "MyDrive" / "FLFrameWork"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

Choose the dataset, method and experiment settings; change the comparison name whenever you start a different experiment.


In [ ]:
from configuration import configuration

config = configuration()
config.dataset = "mnist"
config.method = "fedavg"
config.num_of_clients = 31
config.rounds = 40
config.local_epochs = 5
config.batch_size = 64
config.fairmedian_batch_size = 64
config.lr = 0.01
config.momentum = 0.5 if config.dataset == "mnist" else 0.9
config.dirichlet_alpha = 0.5
config.seed = 42


# FedCDA starts history search after these warmup rounds; 50 means a 40-round run uses warmup only.
config.fedcda_warmup = 50

config.model = {
    "mnist": "cnn_mnist",
    "cifar10": "cnn_cifar10",
    "tinyimagenet": "resnet18",
}[config.dataset]

COMPARISON_NAME = "experiment_1"
folder_name = f"{COMPARISON_NAME}_{config.dataset}"
RESULTS_FOLDER = PROJECT_ROOT / "Results" / folder_name
SAVED_RESULTS = DRIVE_ROOT / "Results" / folder_name
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)
SAVED_RESULTS.mkdir(parents=True, exist_ok=True)


# Restore other methods from this comparison so the analysis can include runs from earlier Colab sessions.
shutil.copytree(SAVED_RESULTS, RESULTS_FOLDER, dirs_exist_ok=True)

print("Device:", config.device)
print("Method:", config.method, "Dataset:", config.dataset)
print("Saved CSV folder:", SAVED_RESULTS)

Download Tiny ImageNet onto the Colab disk when selected; MNIST and CIFAR-10 are downloaded automatically by main.py.


In [ ]:
if config.dataset == "tinyimagenet":
    from torchvision.datasets.utils import download_and_extract_archive

    tiny_folder = PROJECT_ROOT / "data" / "tiny-imagenet-200"
    if not (tiny_folder / "train").is_dir():
        download_and_extract_archive(
            "https://cs231n.stanford.edu/tiny-imagenet-200.zip",
            download_root=str(PROJECT_ROOT / "data"),
        )

    print("Tiny ImageNet:", tiny_folder)
else:
    print(config.dataset, "will download when training starts.")

Run the selected method, measure its elapsed time and copy its CSV to Drive when Python finishes or reports an interruption.


In [ ]:
import copy
import time
from main import main


# A fresh copy avoids retaining algorithm state if this training cell is run again.
run_config = copy.deepcopy(config)
started = time.perf_counter()

try:
    metrics = main(run_config, output_directory=RESULTS_FOLDER)
finally:
    csv_path = RESULTS_FOLDER / f"{run_config.method.lower()}.csv"
    if csv_path.exists():
        try:
            shutil.copy2(csv_path, SAVED_RESULTS / csv_path.name)
            print("Copied CSV to Drive:", SAVED_RESULTS / csv_path.name)
        except OSError as error:
            print("Drive copy failed; the CSV is still on the Colab disk:", csv_path)
            print(error)

    print("Elapsed minutes:", round((time.perf_counter() - started) / 60, 2))

Use the plotting cells from Analysis.ipynb for this comparison and copy their numbered PDF folder to Drive.


In [ ]:
import json

csv_files = sorted(RESULTS_FOLDER.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No method CSVs found in {RESULTS_FOLDER}")

saved_figures = DRIVE_ROOT / "CellResults" / RESULTS_FOLDER.name
local_figures = PROJECT_ROOT / "JPNotebooks" / "CellResults" / RESULTS_FOLDER.name
if saved_figures.exists():
    shutil.copytree(saved_figures, local_figures, dirs_exist_ok=True)

notebook = json.loads((PROJECT_ROOT / "JPNotebooks" / "Analysis.ipynb").read_text())
code_cells = [
    cell for cell in notebook["cells"]
    if cell["cell_type"] == "code"
    and cell["metadata"].get("dataset") == config.dataset
    and cell["metadata"].get("analysis_role") != "load"
]

if len(code_cells) != 3:
    raise ValueError(f"Expected three analysis cells for {config.dataset}.")


# Supply this comparison's CSVs in place of the analysis notebook's editable loading cell.
analysis_data = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "RESULTS_FOLDER": RESULTS_FOLDER,
    "csv_data": {path.name: pd.read_csv(path) for path in csv_files},
}

for cell in code_cells:
    exec("".join(cell["source"]), analysis_data)

shutil.copytree(local_figures, saved_figures, dirs_exist_ok=True)
print("Saved PDFs:", saved_figures / analysis_data["result_folder"].name)

To run another method, change `config.method` in the settings cell and rerun from that cell onward, keeping the same comparison name. Choose a new comparison name when changing the experiment settings. Each dataset has its own folder, and rerunning a method replaces that method's CSV.

For GitHub updates, stop training, push your local changes, restart the Colab Python session, and run setup again. The source cell pulls the new commit. A restart discards in-memory training state; this framework has no checkpoint/resume support. Avoid editing the same source files independently in Colab: `git pull --ff-only` deliberately stops on conflicting local edits.

FairMedian uses one minibatch per client and needs an odd client count. Its local-epoch setting does not apply. FairRFL currently keeps the authors' hardcoded local momentum of 0.9, including on MNIST; setting `config.momentum` does not override it.
